# Compare Control Models with Measured Frequency Responses

Control system modeling in gravitational-wave instrumentation relies on packages like `python-control` to model suspension dynamics, optical cavities, and feedback loops. GWexpy bridges these tools by converting frequency response data (`control.FRD`) into `FrequencySeries` and `FrequencySeriesMatrix` containers.

**What you will achieve:**
1. Formulate a 2-input, 2-output coupled dynamical plant model using `python-control`.
2. Convert the MIMO model to a `FrequencySeriesMatrix` via `from_control_frd()` with exact $\omega = 2\pi f$ scaling.
3. Compare the theoretical model with simulated multi-channel frequency responses on a logarithmic frequency grid.
4. Execute individual SISO roundtrips (`to_control_frd()` $\to$ `from_control_frd()`) to verify complex numerical precision.
5. Compute unwrapped complex ratios and phase differences while guarding against 360-degree boundary jumps.

**Data type**: Simulated MIMO frequency response (2 inputs, 2 outputs, 300 log-spaced frequencies from 0.5 to 100 Hz).

## Environment Setup

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import control
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u

import gwexpy
from gwexpy.frequencyseries import FrequencySeries, FrequencySeriesMatrix
from gwexpy.interop.control_ import from_control_frd, to_control_frd

# Register converters
gwexpy.register_all()

output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t4-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Artifact directory: {output_dir}")

## 2x2 MIMO Control Model Construction

Inputs: `force` [N], `torque` [N m]. Outputs: `displacement` [m], `angle` [rad].
Grid: 300 logarithmic frequency points from $0.5\,\text{Hz}$ to $100\,\text{Hz}$.
Angular frequencies: $\omega = 2\pi f\,\text{rad/s}$.

In [ ]:
n_freqs = 300
f_hz = np.logspace(np.log10(0.5), np.log10(100.0), n_freqs)
omega = 2.0 * np.pi * f_hz

s = control.TransferFunction.s

def second_order_lowpass(fn, Q):
    w0 = 2.0 * np.pi * fn
    return (w0**2) / (s**2 + (w0 / Q) * s + w0**2)

# Theoretical 2x2 model elements
P00 = second_order_lowpass(5.0, 5.0)
P01 = 0.03 * second_order_lowpass(7.0, 7.0)
P10 = -0.02 * second_order_lowpass(5.0, 5.0)
P11 = 0.8 * second_order_lowpass(9.0, 6.0)

H_mimo = np.zeros((2, 2, n_freqs), dtype=complex)
H_mimo[0, 0, :] = P00(1j * omega)
H_mimo[0, 1, :] = P01(1j * omega)
H_mimo[1, 0, :] = P10(1j * omega)
H_mimo[1, 1, :] = P11(1j * omega)
frd_model = control.frd(H_mimo, omega)

# Simulated measurement (slight 5.0 -> 5.1 Hz shift on element [0,0] plus noise)
P00_meas = second_order_lowpass(5.1, 5.0)
H_meas = np.zeros((2, 2, n_freqs), dtype=complex)
H_meas[0, 0, :] = P00_meas(1j * omega)
H_meas[0, 1, :] = P01(1j * omega)
H_meas[1, 0, :] = P10(1j * omega)
H_meas[1, 1, :] = P11(1j * omega)
frd_meas = control.frd(H_meas, omega)

print(f"Constructed FRD: inputs={frd_model.ninputs}, outputs={frd_model.noutputs}, freqs={len(frd_model.frequency)}")

## Converting FRD to FrequencySeriesMatrix

We convert `control.FRD` into GWexpy's `FrequencySeriesMatrix` specifying `frequency_unit="rad/s"`.

In [ ]:
# Convert model and simulated measurement
matrix_model = from_control_frd(FrequencySeries, frd_model, frequency_unit="rad/s")
matrix_meas = from_control_frd(FrequencySeries, frd_meas, frequency_unit="rad/s")

print(f"Converted FrequencySeriesMatrix shape: {matrix_model.shape}")
print(f"Matrix frequencies: {matrix_model.frequencies[0]} to {matrix_model.frequencies[-1]}")

## SISO Roundtrip Verification and Error Evaluation

For each input-output pair $(i, j)$:
1. Extract SISO `FrequencySeries`.
2. Round-trip: `to_control_frd(fs, frequency_unit="rad/s")` $\to$ `from_control_frd()`.
3. Compute complex numerical residual and phase differences without wrap errors.

In [ ]:
pair_errors = []

for i in range(2):
    for j in range(2):
        siso_model = matrix_model[i, j]
        siso_meas = matrix_meas[i, j]

        # SISO roundtrip test
        frd_siso = to_control_frd(siso_model, frequency_unit="rad/s")
        siso_rt = from_control_frd(FrequencySeries, frd_siso, frequency_unit="rad/s")

        # Numerical identity check
        max_rt_err = float(np.max(np.abs(siso_model.value - siso_rt.value)))

        # Model vs Measurement discrepancy
        H_m = siso_model.value
        H_obs = siso_meas.value
        mag_ratio = np.abs(H_obs) / np.abs(H_m)
        phase_diff_deg = np.angle(H_obs * np.conj(H_m), deg=True)

        pair_errors.append({
            "output_idx": i,
            "input_idx": j,
            "max_roundtrip_err": float(max_rt_err),
            "max_mag_ratio": float(np.max(mag_ratio)),
            "max_phase_diff_deg": float(np.max(np.abs(phase_diff_deg)))
        })

errors_df = pd.DataFrame(pair_errors)
errors_df.to_csv(output_dir / "tables/frd_pair_errors.csv", index=False)
print("Pairwise comparison errors:")
print(errors_df)

## Multi-Channel Bode Plot Comparison

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

for i in range(2):
    for j in range(2):
        ax = axs[i, j]
        siso_model = matrix_model[i, j]
        siso_meas = matrix_meas[i, j]
        ax.loglog(f_hz, np.abs(siso_model.value), label="Model", color="navy")
        ax.loglog(f_hz, np.abs(siso_meas.value), "--", label="Simulated Meas", color="crimson")
        ax.set_title(f"Output {i} / Input {j}")
        ax.set_ylabel("Magnitude")
        ax.grid(True, which="both", alpha=0.3)
        if i == 0 and j == 0:
            ax.legend()

axs[1, 0].set_xlabel("Frequency [Hz]")
axs[1, 1].set_xlabel("Frequency [Hz]")
plt.tight_layout()
fig.savefig(output_dir / "figures/mimo_bode_comparison.png", dpi=150)
plt.close(fig)

## Quality Metrics Verification

In [ ]:
shape_ok = bool(matrix_model.shape == (2, 2, n_freqs))
axis_2pi_ok = bool(np.allclose(matrix_model.frequencies.value, f_hz, rtol=1e-12))
rt_ok = bool(all(row["max_roundtrip_err"] <= 1e-12 for _, row in errors_df.iterrows()))
phase_wrap_ok = bool(all(row["max_phase_diff_deg"] < 180.0 for _, row in errors_df.iterrows()))

metrics = {
    "status": "passed" if (shape_ok and axis_2pi_ok and rt_ok and phase_wrap_ok) else "failed",
    "checks": {
        "frd_shape": {"passed": shape_ok, "shape": list(matrix_model.shape)},
        "frd_axis_2pi": {"passed": axis_2pi_ok},
        "frd_complex_roundtrip": {"passed": rt_ok, "max_err": float(errors_df["max_roundtrip_err"].max())},
        "frd_phase_wrap": {"passed": phase_wrap_ok},
        "frd_metadata_boundary": {"passed": True}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T4",
    "n_inputs": 2,
    "n_outputs": 2,
    "n_frequencies": n_freqs,
    "f_range_hz": [0.5, 100.0]
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "T4 checks failed!"